In [1]:
# ============ CELL 1: SETUP AND GPU CHECK ============
print("="*60)
print("STEP 1: SETUP AND GPU CHECK")
print("="*60)

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU Memory: {total_memory:.2f} GB")
else:
    print("WARNING: CUDA not available. Training will be slow on CPU.")

# Clear cache
torch.cuda.empty_cache()
print("GPU cache cleared")
print()

STEP 1: SETUP AND GPU CHECK
PyTorch version: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
GPU Memory: 23.99 GB
GPU cache cleared



In [2]:
# ============ CELL 2: INSTALL PACKAGES ============
print("="*60)
print("STEP 2: INSTALL PACKAGES")
print("="*60)

# Install required packages
! pip install transformers datasets torch -q

print("Packages installed successfully!")
print()

STEP 2: INSTALL PACKAGES
Packages installed successfully!




[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# ============ CELL 3: IMPORTS ============
print("="*60)
print("STEP 3: IMPORTS")
print("="*60)

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
import re
import math
import os
import gc
from typing import List, Dict

print("All imports successful!")
print()

STEP 3: IMPORTS
All imports successful!



In [4]:
# ============ CELL 4: CONFIGURATION ============
print("="*60)
print("STEP 4: CONFIGURATION - DEEPSEEK-R1")
print("="*60)

config = {
    'model_name': 'deepseek-ai/DeepSeek-R1',  # Changed to DeepSeek-R1
    'max_length': 512,  # Increased for R1
    'max_new_tokens': 100,
    'epochs': 3,
    'batch_size': 1,  # Keep small for memory
    'learning_rate': 2e-5,
    'use_quantization': True,  # Add quantization for 7B model
}

print("DeepSeek-R1 Configuration:")
for key, value in config.items():
    print(f"  {key:20}: {value}")
print()

STEP 4: CONFIGURATION - DEEPSEEK-R1
DeepSeek-R1 Configuration:
  model_name          : deepseek-ai/DeepSeek-R1
  max_length          : 512
  max_new_tokens      : 100
  epochs              : 3
  batch_size          : 1
  learning_rate       : 2e-05
  use_quantization    : True



In [5]:
# ============ CELL 5: SIMPLE REWARD CALCULATOR ============
print("="*60)
print("STEP 5: REWARD CALCULATOR")
print("="*60)

class SimpleRewardCalculator:
    """Simple reward calculator for testing"""
    def __init__(self):
        self.pattern = re.compile(r"<think>(.*?)</think>.*?<answer>(.*?)</answer>", re.DOTALL)
    
    def calculate_reward(self, response: str, ground_truth: str) -> Dict[str, float]:
        """Calculate reward for a response"""
        # Basic format check
        if "<think>" not in response or "<answer>" not in response:
            return {"total": 0.0, "format": 0.0, "accuracy": 0.0}
        
        # Format reward
        format_reward = 0.3
        
        # Try to extract and compare numbers
        try:
            # Find all numbers in response
            response_nums = re.findall(r"\d+", response)
            gt_nums = re.findall(r"\d+", ground_truth)
            
            if response_nums and gt_nums:
                # Check if any number matches
                matches = sum(1 for r in response_nums for g in gt_nums if r == g)
                if matches > 0:
                    accuracy = 0.7
                else:
                    accuracy = 0.2
            else:
                accuracy = 0.1
        except:
            accuracy = 0.1
        
        total = format_reward + accuracy
        
        return {
            "total": total,
            "format": format_reward,
            "accuracy": accuracy
        }

# Test the calculator
calculator = SimpleRewardCalculator()
test_response = "User: What is 2+2?\nAssistant: <think>Adding numbers</think><answer>4</answer>"
test_gt = "4"
reward = calculator.calculate_reward(test_response, test_gt)
print(f"Test reward calculation: {reward}")
print()

STEP 5: REWARD CALCULATOR
Test reward calculation: {'total': 1.0, 'format': 0.3, 'accuracy': 0.7}



In [6]:
# ============ CELL 6: CREATE SIMPLE DATASET ============
print("="*60)
print("STEP 6: CREATE DATASET")
print("="*60)

def create_simple_dataset(num_samples=50):
    """Create a simple math dataset for testing"""
    data = []
    
    for i in range(num_samples):
        # Create simple math problems
        a = random.randint(1, 20)
        b = random.randint(1, 20)
        operation = random.choice(['+', '-', '*'])
        
        if operation == '+':
            problem = f"What is {a} + {b}?"
            answer = a + b
        elif operation == '-':
            problem = f"What is {a} - {b}?"
            answer = a - b
        else:  # '*'
            problem = f"What is {a} × {b}?"
            answer = a * b
        
        # Format with think/answer tags
        text = f"User: {problem}\nAssistant: <think>Calculating {a} {operation} {b}</think><answer>{answer}</answer>"
        
        data.append({
            'problem': problem,
            'answer': str(answer),
            'text': text
        })
    
    return data

# Create dataset
print("Creating dataset...")
raw_data = create_simple_dataset(30)  # Small dataset for testing
print(f"Created {len(raw_data)} samples")

# Show examples
print("\nSample data:")
for i in range(min(2, len(raw_data))):
    print(f"  {raw_data[i]['text'][:80]}...")
print()

STEP 6: CREATE DATASET
Creating dataset...
Created 30 samples

Sample data:
  User: What is 7 + 16?
Assistant: <think>Calculating 7 + 16</think><answer>23</an...
  User: What is 3 - 11?
Assistant: <think>Calculating 3 - 11</think><answer>-8</an...



In [7]:
# ============ CELL 7: LOAD DEEPSEEK-R1 MODEL ============
print("="*60)
print("STEP 7: LOAD DEEPSEEK-R1 MODEL")
print("="*60)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Clear memory
torch.cuda.empty_cache()

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Configure quantization for memory efficiency
if config.get('use_quantization', True) and torch.cuda.is_available():
    print("Configuring 8-bit quantization...")
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
        llm_int8_has_fp16_weight=True,
        bnb_4bit_compute_dtype=torch.float16
    )
else:
    quantization_config = None
    print("Using full precision (no quantization)")

# Load DeepSeek-R1 tokenizer
print(f"\nLoading DeepSeek-R1 tokenizer...")
try:
    tokenizer = AutoTokenizer.from_pretrained(
        config['model_name'],
        trust_remote_code=True
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    print(f"✓ Tokenizer loaded")
    print(f"  Vocabulary size: {tokenizer.vocab_size}")
    print(f"  Pad token: {tokenizer.pad_token}")
    
except Exception as e:
    print(f"Error loading tokenizer: {e}")
    print("Falling back to GPT-2 tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

# Load DeepSeek-R1 model
print(f"\nLoading DeepSeek-R1 model...")
try:
    if quantization_config:
        model = AutoModelForCausalLM.from_pretrained(
            config['model_name'],
            quantization_config=quantization_config,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        print("✓ Model loaded with 8-bit quantization")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            config['model_name'],
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
            trust_remote_code=True
        )
        print("✓ Model loaded with full precision")
    
    # Print model info
    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Total parameters: {total_params:,}")
    
    # Check if model is on GPU
    if hasattr(model, 'hf_device_map'):
        print(f"  Device map: {model.hf_device_map}")
    else:
        print(f"  Model device: {next(model.parameters()).device}")
    
except Exception as e:
    print(f"Error loading DeepSeek-R1: {e}")
    print("\nFalling back to smaller model...")
    
    # Fallback to a smaller model
    config['model_name'] = 'deepseek-ai/deepseek-math-7b-base'
    print(f"Trying: {config['model_name']}")
    
    try:
        model = AutoModelForCausalLM.from_pretrained(
            config['model_name'],
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
        print("✓ Fallback model loaded")
    except Exception as e2:
        print(f"Fallback failed: {e2}")
        print("Using GPT-2 as last resort...")
        config['model_name'] = 'gpt2'
        model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
        tokenizer = AutoTokenizer.from_pretrained("gpt2")
        tokenizer.pad_token = tokenizer.eos_token

print(f"\nFinal model: {config['model_name']}")
print("Model loading complete!")
print()

STEP 7: LOAD DEEPSEEK-R1 MODEL
Using device: cuda
Configuring 8-bit quantization...

Loading DeepSeek-R1 tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\user\.conda\envs\gpu_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--deepseek-ai--DeepSeek-R1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json: 0.00B [00:00, ?B/s]

✓ Tokenizer loaded
  Vocabulary size: 128000
  Pad token: <｜end▁of▁sentence｜>

Loading DeepSeek-R1 model...


...

modeling_deepseek.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/deepseek-ai/DeepSeek-R1:
- modeling_deepseek.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
c:\Users\user\.conda\envs\gpu_env\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\user\.conda\envs\gpu_env\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\user\

Error loading DeepSeek-R1: The model is quantized with FineGrainedFP8Config but you are passing a BitsAndBytesConfig config. Please make sure to pass the same quantization config class to `from_pretrained` with different loading attributes.

Falling back to smaller model...
Trying: deepseek-ai/deepseek-math-7b-base


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✓ Fallback model loaded

Final model: deepseek-ai/deepseek-math-7b-base
Model loading complete!



In [9]:
# ============ CELL 7: TOKENIZE DATASET ============
print("="*60)
print("STEP 7: TOKENIZE DATASET")
print("="*60)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(config['model_name'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded: {config['model_name']}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Tokenize dataset
tokenized_data = []
for item in raw_data:
    encoded = tokenizer(
        item['text'],
        truncation=True,
        max_length=config['max_length'],
        padding='max_length',
        return_tensors='pt'
    )
    
    tokenized_data.append({
        'input_ids': encoded['input_ids'][0],
        'attention_mask': encoded['attention_mask'][0],
        'text': item['text'],
        'ground_truth': item['answer']
    })

print(f"Tokenized {len(tokenized_data)} samples")
print(f"Input shape: {tokenized_data[0]['input_ids'].shape}")
print()

STEP 7: TOKENIZE DATASET
Tokenizer loaded: deepseek-ai/deepseek-math-7b-base
Vocabulary size: 100000
Tokenized 30 samples
Input shape: torch.Size([512])



In [10]:
# ============ CELL 8: CREATE DATASET CLASS ============
print("="*60)
print("STEP 8: CREATE DATASET CLASS")
print("="*60)

class MathDataset(Dataset):
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': item['input_ids'].clone().detach(),
            'attention_mask': item['attention_mask'].clone().detach(),
            'text': item['text'],
            'ground_truth': item['ground_truth']
        }

# Create dataset
dataset = MathDataset(tokenized_data)
print(f"Dataset created with {len(dataset)} samples")
print()

STEP 8: CREATE DATASET CLASS
Dataset created with 30 samples



In [11]:
# ============ CELL 9: SIMPLE TRAINER CLASS ============
print("="*60)
print("STEP 9: CREATE TRAINER")
print("="*60)

class SimpleTrainer:
    def __init__(self, model, tokenizer, config):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'])
        self.reward_calculator = SimpleRewardCalculator()
    
    def train_step(self, batch):
        """Simple training step without generation"""
        # Forward pass
        outputs = self.model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['input_ids']  # Use input as labels for language modeling
        )
        
        loss = outputs.loss
        
        # Backward pass
        self.optimizer.zero_grad()
        loss.backward()
        
        # Clip gradients
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        
        # Optimizer step
        self.optimizer.step()
        
        return loss.item()
    
    def generate_and_evaluate(self, batch):
        """Generate responses and calculate rewards"""
        with torch.no_grad():
            self.model.eval()
            
            # Generate response
            outputs = self.model.generate(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                max_new_tokens=config['max_new_tokens'],
                do_sample=True,
                temperature=0.7,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
            
            # Decode responses
            responses = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)
            
            # Calculate rewards
            rewards = []
            for response, gt in zip(responses, batch['ground_truth']):
                reward = self.reward_calculator.calculate_reward(response, gt)['total']
                rewards.append(reward)
            
            avg_reward = sum(rewards) / len(rewards) if rewards else 0.0
            
            return responses, avg_reward

print("Trainer class created")
print()

STEP 9: CREATE TRAINER
Trainer class created



In [13]:
# ============ CELL 11: SAVE DEEPSEEK-R1 MODEL ============
print("="*60)
print("STEP 11: SAVE DEEPSEEK-R1 MODEL")
print("="*60)

# Save model
save_dir = "./deepseek_r1_trained"
os.makedirs(save_dir, exist_ok=True)

try:
    if not hasattr(model, 'hf_quantizer'):  # Can't save quantized models directly
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"✓ Model saved to: {save_dir}")
    else:
        # For quantized models, save state dict
        torch.save(model.state_dict(), os.path.join(save_dir, "model_weights.pth"))
        tokenizer.save_pretrained(save_dir)
        print(f"✓ Quantized model weights saved to: {save_dir}")
    
except Exception as e:
    print(f"Error saving model: {e}")
    print("Saving tokenizer only...")
    tokenizer.save_pretrained(save_dir)

# Test the trained model
print("\nFinal DeepSeek-R1 Test:")
print("-" * 50)

test_questions = [
    "Calculate 12 + 9",
    "What is 25 - 13?",
    "Solve 7 × 8",
    "Find the result of 45 + 27",
    "Compute 63 - 28",
]

model.eval()
for question in test_questions:
    try:
        prompt = f"User: {question}\n\nAssistant: "
        inputs = tokenizer(prompt, return_tensors="pt")
        
        if not hasattr(model, 'hf_device_map'):
            inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id
            )
        
        response = tokenizer.decode(output[0], skip_special_tokens=True)
        answer = response[len(prompt):].strip()
        
        print(f"Q: {question}")
        print(f"A: {answer[:100]}...\n")
        
    except Exception as e:
        print(f"Error on '{question}': {e}\n")

print("="*60)
print("DEEPSEEK-R1 TRAINING FINISHED!")
print("="*60)

STEP 11: SAVE DEEPSEEK-R1 MODEL
✓ Model saved to: ./deepseek_r1_trained

Final DeepSeek-R1 Test:
--------------------------------------------------
Error on 'Calculate 12 + 9': Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument index in method wrapper_CUDA__index_select)

Error on 'What is 25 - 13?': Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument index in method wrapper_CUDA__index_select)

Error on 'Solve 7 × 8': Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument index in method wrapper_CUDA__index_select)

Error on 'Find the result of 45 + 27': Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument index in method wrapper_CUDA__index_select)

Error on 'Compute 63 - 